- 0 = Speed limit
- 1 = Prohibited direction
- 2 = Other prohibited
- 3 = Mandatory direction
- 4 = Crossing / road user warning
- 5 = Road hazard / condition warning
- 6 = Traffic control / other regulatory

In [20]:
!pip install torch 
!pip install pandas matplotlib pillow scikit-learn

In [21]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

Using device: cpu


In [22]:
class_names = {
    0: "Speed limit",
    1: "Prohibited direction",
    2: "Other prohibited",
    3: "Mandatory direction",
    4: "Crossing / road user warning",
    5: "Road hazard / condition warning",
    6: "Traffic control / other regulatory",
}

NUM_CLASSES = len(class_names)
#class_names

In [23]:
MODEL_PATH = Path("traffic_sign_grouped_resnet18.pth") #model for 7 classes
# MODEL_PATH = Path("traffic_sign_model_58class.pth") #model for 50 classes


if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "Could not find traffic_sign_grouped_resnet18.pt. "
        "Put the saved model file in the same folder as this demo notebook."
    )

# Load the checkpoint on CPU.
# weights_only=False helps newer PyTorch versions load a full checkpoint dictionary.
try:
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

NUM_CLASSES = checkpoint.get("num_classes", NUM_CLASSES)
class_names = checkpoint.get("class_names", class_names)

# Sometimes dictionary keys can load as strings, so convert them back to integers.
class_names = {int(k): v for k, v in class_names.items()}

IMG_SIZE = checkpoint.get("img_size", 224)
imagenet_mean = checkpoint.get("imagenet_mean", [0.485, 0.456, 0.406])
imagenet_std = checkpoint.get("imagenet_std", [0.229, 0.224, 0.225])

# Rebuild the same ResNet18 structure used during training.
model = models.resnet18(weights=None)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.30),
    nn.Linear(in_features, NUM_CLASSES)
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

# print("Loaded model successfully.")
# print("Number of classes:", NUM_CLASSES)
# print("Classes:", class_names)

FileNotFoundError: Could not find traffic_sign_grouped_resnet18.pt. Put the saved model file in the same folder as this demo notebook.

In [ ]:
demo_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

In [ ]:
DEMO_IMAGE_DIR = Path("demo_images")

if not DEMO_IMAGE_DIR.exists():
    raise FileNotFoundError(
        "Could not find the demo_images folder."
    )

image_extensions = [".jpg", ".jpeg", ".png"]

demo_image_paths = sorted([
    p for p in DEMO_IMAGE_DIR.rglob("*")
    if p.suffix.lower() in image_extensions
])

print("Number of demo images found:", len(demo_image_paths))

demo_image_paths[:10]

Number of demo images found: 10


[PosixPath('demo_images/1.jpg'),
 PosixPath('demo_images/10.png'),
 PosixPath('demo_images/2.jpg'),
 PosixPath('demo_images/3.jpg'),
 PosixPath('demo_images/4.png'),
 PosixPath('demo_images/5.png'),
 PosixPath('demo_images/6.png'),
 PosixPath('demo_images/7.png'),
 PosixPath('demo_images/8.png'),
 PosixPath('demo_images/9.png')]

In [ ]:
LABELS_PATH = Path("demo_labels.csv")

if LABELS_PATH.exists():
    demo_labels_df = pd.read_csv(LABELS_PATH)
    print("Loaded demo_labels.csv")
    display(demo_labels_df.head())
else:
    # This creates a starter CSV that you can fill in manually.
    starter_df = pd.DataFrame({
        "filename": [p.name for p in demo_image_paths],
        "true_group_id": [""] * len(demo_image_paths),
    })
    starter_df.to_csv(LABELS_PATH, index=False)
    demo_labels_df = None
    print("demo_labels.csv was not found")

Loaded demo_labels.csv


,filename,true_group_id
0,1.jpg,6
1,2.jpg,5
2,3.jpg,0
3,4.png,4
4,5.png,4


In [ ]:
@torch.no_grad()
def predict_image(image_path, model, transform):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)

    outputs = model(tensor)
    probabilities = torch.softmax(outputs, dim=1).squeeze(0).numpy()

    predicted_class = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_class])

    return predicted_class, confidence, probabilities

In [ ]:
results = []

for image_path in demo_image_paths:
    pred_class, confidence, probabilities = predict_image(image_path, model, demo_transform)

    results.append({
        "filename": image_path.name,
        "image_path": str(image_path),
        "predicted_group_id": pred_class,
        "predicted_group_name": class_names[pred_class],
        "confidence": confidence,
    })

results_df = pd.DataFrame(results)

if LABELS_PATH.exists():
    labels_df = pd.read_csv(LABELS_PATH)

    # Only merge labels if true_group_id has been filled in.
    if "true_group_id" in labels_df.columns and labels_df["true_group_id"].notna().any():
        labels_df["true_group_id"] = pd.to_numeric(labels_df["true_group_id"], errors="coerce")
        results_df = results_df.merge(labels_df, on="filename", how="left")
        results_df["true_group_name"] = results_df["true_group_id"].map(class_names)

        if results_df["true_group_id"].notna().any():
            results_df["correct"] = results_df["predicted_group_id"] == results_df["true_group_id"]

results_df

In [ ]:
for _, row in results_df.iterrows():
    image = Image.open(row["image_path"]).convert("RGB")

    plt.figure(figsize=(4, 4))
    plt.imshow(image)

    title = f"Predicted: {row['predicted_group_name']}\nConfidence: {row['confidence']:.2%}"

    if "true_group_name" in results_df.columns and pd.notna(row.get("true_group_name")):
        title = f"True: {row['true_group_name']}\n" + title

    plt.title(title)
    plt.axis("off")
    plt.show()